## Objective
This notebook covers two tasks:
- Generate a synthetic dataset based on Gardner's Multiple Intelligences theory
- Perform exploratory data analysis (EDA) on the generated dataset

## How to use this notebook with real data

This notebook is designed to work in two modes:

| Mode | How to activate |
|---|---|
| **Synthetic (default)** | Just run the notebook as-is |
| **Real data** | Place a CSV file named `real_data.csv` inside the `data/` folder with the same column names as the synthetic dataset. The notebook will detect it automatically and skip data generation. 

**Expected columns for real data:**

```
student_name, linguistic_score, logical_math_score, spatial_score,
bodily_kinesthetic_score, interpersonal_score, intrapersonal_score,
emotional_regulation, engagement_frequency
```

All score columns must be numeric values between 0 and 10.

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker

# Reproducibility
SEED = 42
np.random.seed(SEED)

DATA_DIR = "../data"
SYNTHETIC_PATH = os.path.join(DATA_DIR, "synthetic_dataset.csv")
REAL_DATA_PATH = os.path.join(DATA_DIR, "real_data.csv")

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("Setup complete.")

Setup complete.


## 2. Data Source Detection

The notebook automatically checks whether a real dataset exists in `data/real_data.csv`.  
If found, it uses real data. Otherwise, it generates a synthetic dataset.

In [2]:
USE_REAL_DATA = os.path.exists(REAL_DATA_PATH)

if USE_REAL_DATA:
    print("***Real data file detected at:", REAL_DATA_PATH)
    print("***Skipping synthetic data generation.***")
else:
    print("***No real data file found.***")
    print("***Synthetic dataset will be generated automatically.***")

***No real data file found.***
***Synthetic dataset will be generated automatically.***


## 3. Synthetic Data Generation

### Design rationale

Profiles are generated with **internal coherence**, meaning scores across dimensions are not fully independent. 
This reflects pedagogical reality: a child with high logical-mathematical intelligence often shows lower bodily-kinesthetic engagement in traditional classroom settings, while a child with high interpersonal scores tends to show higher engagement frequency.

Four **base profile archetypes** were defined based on Gardner's theory to seed the generation:

| Archetype | Description |
|---|---|
| **Analytical** | High logical-math and spatial; lower bodily-kinesthetic and interpersonal |
| **Communicative** | High linguistic and interpersonal; moderate other dimensions |
| **Kinesthetic** | High bodily-kinesthetic and engagement; lower linguistic and logical-math |
| **Balanced** | Moderate scores across all dimensions; moderate socioemotional indicators |

Each student is generated from one of these archetypes with added Gaussian noise to simulate natural variation.

> **Note:** These archetypes are used only as a generation mechanism.  
> The K-Means model covered later will discover clusters **independently**, without knowledge of these labels.

For this project, some design decisions were made:
- "High" scores were considered as 8.5
- "Moderate" as 7.0
- "Low" as 5.5
- These values were chosen to reflect the Brazilian school grading scale (0–10),
  where scores above 7.0 are generally considered good performance
- The variation will be defined by the NOISE_STD parameter
- These values can be adjusted according to the user's needs

In [5]:
if not USE_REAL_DATA:
    fake = Faker('pt_BR')
    Faker.seed(SEED)

    N_STUDENTS = 120
    NOISE_STD = 0.8  # Standard deviation of Gaussian noise added to each score

    # Base profile archetypes
    # Format: [linguistic, logical_math, spatial, bodily_kinesthetic,
    #          interpersonal, intrapersonal, emotional_regulation, engagement_frequency]
    archetypes = {
        #                ling   log   spat  body  inter intra emot  engag
        "analytical":    [6.0,  8.5,  8.5,  3.0,  3.0,  6.0,  6.0,  6.0],
        "communicative": [8.5,  6.0,  6.0,  6.0,  8.5,  6.0,  6.0,  8.5],
        "kinesthetic":   [3.0,  3.0,  6.0,  8.5,  6.0,  6.0,  3.0,  8.5],
        "balanced":      [6.0,  6.0,  6.0,  6.0,  6.0,  6.0,  6.0,  6.0],
    }

    # Proportional distribution across archetypes
    archetype_weights = [0.15, 0.20, 0.25, 0.40]
    archetype_names = list(archetypes.keys())

    columns = [
        "student_name",
        "linguistic_score",
        "logical_math_score",
        "spatial_score",
        "bodily_kinesthetic_score",
        "interpersonal_score",
        "intrapersonal_score",
        "emotional_regulation",
        "engagement_frequency",
        "generated_archetype"  # kept for transparency — NOT used in modeling
    ]

    rows = []
    chosen_archetypes = np.random.choice(
        archetype_names, size=N_STUDENTS, p=archetype_weights
    )

    for archetype in chosen_archetypes:
        base_scores = archetypes[archetype]
        noisy_scores = np.array(base_scores) + np.random.normal(0, NOISE_STD, len(base_scores))

        row = [fake.name()] + list(noisy_scores) + [archetype]
        rows.append(row)

    df = pd.DataFrame(rows, columns=columns)

    # Save to CSV
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_csv(SYNTHETIC_PATH, index=False)

    print(f"***Synthetic dataset generated: {N_STUDENTS} students, {len(columns)} columns")
    print(f"***Saved to: {SYNTHETIC_PATH}")
    print(f"\nArchetype distribution:")
    print(df['generated_archetype'].value_counts())

NameError: name 'TRUE' is not defined